## Import Functions

In [ ]:
from __future__ import annotations

# === Standard Library ===
import logging
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from typing import Any, Iterable, Optional, Type, Union
import datetime

# === Third-Party Libraries ===
import pandas as pd
import numpy as np
from numpy.typing import NDArray
import matplotlib.pyplot as plt
import jiwer

import torch
from torch import Tensor

import soundfile as sf

from huggingface_hub import hf_hub_download

import yaml
import phonemizer
from phonemizer.backend import EspeakBackend

from nltk.cluster import cosine_distance
from nltk.tokenize import word_tokenize

from sentence_transformers import SentenceTransformer, util
from transformers import GPT2Tokenizer, GPT2LMHeadModel, Wav2Vec2Model, Wav2Vec2Processor
import math

import IPython.display as ipd

# === Pymoo Optimization Framework ===
from _pymoo_optimizer import PymooOptimizer

from pymoo.algorithms.base.genetic import GeneticAlgorithm
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.core.evaluator import Evaluator
from pymoo.core.population import Population
from pymoo.core.problem import Problem
from pymoo.core.termination import NoTermination
from pymoo.operators.crossover.sbx import SBX
from pymoo.operators.mutation.pm import PM
from pymoo.operators.sampling.rnd import FloatRandomSampling
from pymoo.problems.static import StaticProblem

# === StyleTTS2 Local Modules ===
from models import *
from utils import *
from text_utils import TextCleaner
from Utils.PLBERT.util import load_plbert
from Modules.diffusion.sampler import DiffusionSampler, ADPM2Sampler, KarrasSchedule

# === Import from Files ===
from _helper import length_to_mask, addNumbersPattern, adjustInterpolationVector, interpolateWithScalar, text_naturalness_from_ppl
from _pymoo_optimizer import PymooOptimizer
from _styletts2 import StyleTTS2
from _asr_model import AutomaticSpeechRecognitionModel

%cd ..

## Main Function

### Initializes Values

In [ ]:
ACTIVE_OBJECTIVES = [
    # --- Quality & Intelligibility ---
    "utmos",              # Maximizes MOS Quality (Recommended)
    "wer",                # Minimizes Word Error Rate (Requires ASR)
    # "avg-logprob",        # Maximizes ASR Confidence

    # --- Speaker Similarity (Voice Cloning) ---
    # "wav2vec-target",     # Cosine Similarity to Target Voice (Strongest)
    # "s-bert-target",      # Semantic Similarity to Target (Content)
    # "text-embedding-target", # Embedding Similarity to Target

    # --- Ground Truth Retention (regularization) ---
    "wav2vec-gt",         # Keep voice similar to original
    # "s-bert-gt",          # Keep content similar to original

    # --- Vector Constraints ---
    # "l1",                 # Minimize L1 Norm (Sparsity)
    # "l2",                 # Minimize L2 Norm (Keep values small)
]

diffusion_steps = 5
embedding_scale = 1

interpolation_percentage = 0.4 # How much of Target to be used, small interpolation_percentage means more of ground_truth (Minimization)

name_gt = "ground_truth"
text_gt = "I think the NFL is lame and boring"

name_target = "target"
text_target = "The Seattle Seahawks are the best Team in the world"

device = 'cuda' if torch.cuda.is_available() else 'cpu'
noise = torch.randn(1, 1, 256).to(device)

### Load Models

In [ ]:
### Required Models ###

tts_model = StyleTTS2()
tts_model.load_models()  # builds self.model and loads self.params
tts_model.load_checkpoints()  # puts params into self.model
tts_model.sample_diffusion()  # builds self.sampler

h_text_gt, h_bert_raw_gt, h_bert_gt, h_text_target, h_bert_raw_target, h_bert_target, input_lengths, text_mask, style_vector_acoustic, style_vector_prosodic = tts_model.extract_mixed_embeddings(text_gt, text_target, noise, embedding_scale, diffusion_steps)

audio_gt = tts_model.inference_after_interpolation(input_lengths, text_mask, h_bert_gt, h_text_gt, style_vector_acoustic, style_vector_prosodic)
audio_target = tts_model.inference_after_interpolation(input_lengths, text_mask, h_bert_target, h_text_target, style_vector_acoustic, style_vector_prosodic)

asr_model = AutomaticSpeechRecognitionModel("tiny", device=device)


### Conditional Models ####

if "text-embedding-target" in ACTIVE_OBJECTIVES or "text-embedding-gt" in ACTIVE_OBJECTIVES:
    embedding_model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2", device=device)
    embedding_model.eval()

    text_embedding_gt = embedding_model.encode(text_gt, convert_to_tensor=True, normalize_embeddings=True)
    text_embedding_target = embedding_model.encode(text_target, convert_to_tensor=True, normalize_embeddings=True)

if "utmos" in ACTIVE_OBJECTIVES:
    utmos_model = torch.jit.load(
        hf_hub_download(
            repo_id="balacoon/utmos",
            filename="utmos.jit",
            repo_type="model",
            local_dir="./"
        ),
        map_location=device
    )
    utmos_model.eval()

if "s-bert-target" in ACTIVE_OBJECTIVES or "s-bert-gt" in ACTIVE_OBJECTIVES:
    sbert_model = SentenceTransformer('all-MiniLM-L6-v2', device=device)
    sbert_model.eval()

    s_bert_embedding_gt = sbert_model.encode(text_gt, convert_to_tensor=True, normalize_embeddings=True)
    s_bert_embedding_target = sbert_model.encode(text_target, convert_to_tensor=True, normalize_embeddings=True)

if "wer" in ACTIVE_OBJECTIVES:
    wer_transformations = jiwer.Compose([
        jiwer.ExpandCommonEnglishContractions(),
        jiwer.RemoveEmptyStrings(),
        jiwer.ToLowerCase(),
        jiwer.RemoveMultipleSpaces(),
        jiwer.Strip(),
        jiwer.RemovePunctuation(),
        jiwer.ReduceToListOfListOfWords(),
    ])

if "ppl" in ACTIVE_OBJECTIVES:
    perplexity_model = GPT2LMHeadModel.from_pretrained("gpt2").to(device)
    perplexity_model.eval()
    perplexity_tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

if "wav2vec-gt" in ACTIVE_OBJECTIVES or "wav2vec-target" in ACTIVE_OBJECTIVES:
    wav2vec_processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-large-960h-lv60-self")
    wav2vec_model = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-large-960h-lv60-self").to(device)
    wav2vec_model.eval()

    with torch.no_grad():
        wav2vec_embedding_gt = torch.mean(wav2vec_model(**wav2vec_processor(audio_gt, return_tensors="pt", sampling_rate=16000).to(device)).last_hidden_state, dim=1)
        wav2vec_embedding_target = torch.mean(wav2vec_model(**wav2vec_processor(audio_target, return_tensors="pt", sampling_rate=16000).to(device)).last_hidden_state, dim=1)

### Optimizer

#### Initialize Optimizer

In [ ]:
pop_size = 100
num_generations = 100
num_objectives = ACTIVE_OBJECTIVES.__len__()

# Solution Shape = (s,p) with p = number of phonemes, s = size of vector per phoneme
phoneme_count = input_lengths.detach().cpu().item()
size_per_phoneme = 8

solution_shape = (phoneme_count, size_per_phoneme)

random_matrix = np.random.rand(size_per_phoneme, 512)
# Alternatives: rng.random((512, s)), np.random.randn(512, s)

random_matrix = torch.from_numpy(random_matrix).to(device).float() # For Matrix Multiplication

# Genetic algorithm parameters
algo_params = {
    "pop_size": pop_size,
}

optimizer = PymooOptimizer(
    bounds=(0, 1),
    algorithm=NSGA2,
    algo_params=algo_params,
    num_objectives=num_objectives,
    solution_shape=solution_shape,
)

#### Loop

In [ ]:
# 1. Initialize a list to hold all records
fitness_history = []
mean_model = []

for gen in range(num_generations):

    gen_scores = {key: [] for key in ACTIVE_OBJECTIVES}

    # 1) Get current population
    population_vectors = optimizer.get_x_current()  # shape: (pop_size, *solution_shape)

    # Debugging
    print(f"=== Generation {gen} ===")

    for j, interpolation_vector in enumerate(population_vectors):

        # Ensure vector is on the correct device
        interpolation_vector = torch.from_numpy(interpolation_vector).to(device).float()
        interpolation_vector = adjustInterpolationVector(interpolation_vector, random_matrix, size_per_phoneme)

        # Interpolate Values
        h_text_mixed = (1.0 - interpolation_vector) * h_text_gt + interpolation_vector * h_text_target
        h_bert_mixed = h_bert_gt

        # Inference
        audio_mixed = tts_model.inference_after_interpolation(
            input_lengths,
            text_mask,
            h_bert_mixed,
            h_text_mixed,
            style_vector_acoustic,
            style_vector_prosodic
        )

        # ASR Analysis
        asr_result, asr_confidence = asr_model.analyzeAudio(audio_mixed)
        asr_text = asr_result["text"]

        current_ind_scores = {}

        # --- Calculate Fitness Functions ---
        if "l1" in ACTIVE_OBJECTIVES:
            # Minimize Interpolation Vector (L1-Norm)
            val = float(interpolation_vector.abs().mean().item())
            gen_scores["l1"].append(val)
            current_ind_scores["L1_Norm"] = val

        if "l2" in ACTIVE_OBJECTIVES:
            # Minimize Interpolation Vector (L2-Norm)
            val = (interpolation_vector ** 2).mean().item()
            gen_scores["l2"].append(val)
            current_ind_scores["L2_Norm"] = val

        if "wer" in ACTIVE_OBJECTIVES:
            # Minimize Word Error Rate
            val = jiwer.wer(
                text_target,
                asr_text,
                reference_transform=wer_transformations,
                hypothesis_transform=wer_transformations,
            )
            val = float(val)
            gen_scores["wer"].append(val)
            current_ind_scores["WER_Score"] = val

        if "s-bert-target" in ACTIVE_OBJECTIVES:
            # 1. Compute Embedding of ASR result
            s_bert_embedding_asr = sbert_model.encode(asr_text, convert_to_tensor=True, normalize_embeddings=True)

            # 2. Compute Similarity (1.0 is best)
            val = 1.0 - util.cos_sim(s_bert_embedding_target, s_bert_embedding_asr).item()

            # 3. Store
            gen_scores["s-bert-target"].append(val)
            current_ind_scores["S-BERT-Target"] = val

        if "s-bert-gt" in ACTIVE_OBJECTIVES:
            # 1. Compute Embedding of ASR result
            s_bert_embedding_asr = sbert_model.encode(asr_text, convert_to_tensor=True, normalize_embeddings=True)

            # 2. Compute Similarity (1.0 is best)
            val = 1 - util.cos_sim(s_bert_embedding_gt, s_bert_embedding_asr).item()
            val = - val

            # 3. Store
            gen_scores["s-bert-gt"].append(val)
            current_ind_scores["S-BERT-Ground-Truth"] = val

        if "text-embedding-gt" in ACTIVE_OBJECTIVES:
            # Maximize Embedding Distance (Ground Truth)
            text_embedding_mixed = embedding_model.encode(asr_text, convert_to_tensor=True, normalize_embeddings=True)
            val = - (1.0 - F.cosine_similarity(text_embedding_gt, text_embedding_mixed, dim=1).item())
            gen_scores["text-embedding-gt"].append(val)
            current_ind_scores["Text-Embedding-Distance-GT"] = val

        if "text-embedding-target" in ACTIVE_OBJECTIVES:
            # Minimize Embedding Distance (Target)
            text_embedding_mixed = embedding_model.encode(asr_text, convert_to_tensor=True, normalize_embeddings=True)
            val = 1.0 - F.cosine_similarity(text_embedding_target, text_embedding_mixed, dim=1).item()
            gen_scores["text-embedding-target"].append(val)
            current_ind_scores["Text-Embedding-Distance-Target"] = val

        if "avg-logprob" in ACTIVE_OBJECTIVES:
            # Maximize ASR Confidence
            val = - asr_confidence
            gen_scores["avg-logprob"].append(val)
            current_ind_scores["Average-Log-Probability"] = val

        if "utmos" in ACTIVE_OBJECTIVES:
            val = - utmos_model(torch.from_numpy(audio_mixed).float().unsqueeze(0).to(next(utmos_model.parameters()).device)).item()
            gen_scores["utmos"].append(val)
            current_ind_scores["UTMOS"] = val

        if "wav2vec-gt" in ACTIVE_OBJECTIVES:
            wav2vec_embedding_mixed = torch.mean(wav2vec_model(**wav2vec_processor(audio_mixed, return_tensors="pt", sampling_rate=16000).to(device)).last_hidden_state, dim=1)
            val = 1 - F.cosine_similarity(wav2vec_embedding_gt, wav2vec_embedding_mixed).item()
            gen_scores["wav2vec-gt"].append(val)
            current_ind_scores["wav2vec-ground-truth"] = val

        if "wav2vec-target" in ACTIVE_OBJECTIVES:
            wav2vec_embedding_mixed = torch.mean(wav2vec_model(**wav2vec_processor(audio_mixed, return_tensors="pt", sampling_rate=16000).to(device)).last_hidden_state, dim=1)
            val = 1 - F.cosine_similarity(wav2vec_embedding_target, wav2vec_embedding_mixed).item()
            gen_scores["wav2vec-target"].append(val)
            current_ind_scores["wav2vec-target"] = val

        # 2. Store individual record
        record = {"Generation": gen, "Individual_ID": j}
        record.update(current_ind_scores)
        fitness_history.append(record)

    # 3. Calculate Means dynamically
    gen_mean = {"Generation": gen}
    fitness_arrays_for_optimizer = []

    # Iterate over ACTIVE_OBJECTIVES to ensure order is preserved for optimizer
    for key in ACTIVE_OBJECTIVES:
        # Convert list to array
        arr = np.array(gen_scores[key], dtype=float)

        # Store mean for dataframe
        gen_mean[f"{key}_Mean"] = np.mean(arr)

        # Prepare list for optimizer
        fitness_arrays_for_optimizer.append(arr)

    mean_model.append(gen_mean)

    # 4. Update Optimizer
    optimizer.assign_fitness(fitness_arrays_for_optimizer)
    optimizer.update()

#### Inference after Interpolation

In [ ]:
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M")

folder_path = f"outputs/h_text/{'_'.join(ACTIVE_OBJECTIVES)}/{timestamp}/"

os.makedirs(folder_path, exist_ok=True)

##### Graph #####

# Convert the history list to a DataFrame
df_all_fitness = pd.DataFrame(fitness_history)

# Convert the median tracking to a DataFrame (easier to plot later)
df_means = pd.DataFrame(mean_model)

# Filters out 'Generation' and grabs everything else (e.g., 'f1_Mean', 'f3_Mean')
fitness_cols = [col for col in df_means.columns if 'Mean' in col and col != 'Generation']

# 2. Create dynamic labels (optional, you can map them if you want specific names)
# If you want specific names, map them here. Otherwise, it defaults to the column name.
name_map = {
    "f1_Mean": "f1 (L1 Norm)",
    "f2_Mean": "f2 (Emb Dist GT)",
    "f3_Mean": "f3 (Emb Dist Target)",
    "f4_Mean": "f4 (ASR Confidence)"
}

# 3. Setup Plot
fig, axs = plt.subplots(len(fitness_cols), 1, figsize=(14, 5 * len(fitness_cols)))
if len(fitness_cols) == 1: axs = [axs] # Handle single plot edge case
fig.suptitle("Evolution of Fitness Objectives (Mean)", fontsize=16)

# 4. Loop through the EXISTING columns
for j, col_name in enumerate(fitness_cols):

    x_data = df_means['Generation']
    y_data = df_means[col_name]

    # Get a nice label if available, otherwise use column name
    plot_label = name_map.get(col_name, col_name)
    color = 'blue' if j % 2 == 0 else 'green' # Alternate colors

    axs[j].plot(x_data, y_data,
                color=color,
                linestyle='--',
                alpha=0.6,
                label="Population Mean")

    axs[j].set_title(plot_label)
    axs[j].set_xlabel("Generation")
    axs[j].set_ylabel("Fitness")
    axs[j].grid(True, alpha=0.3)
    axs[j].legend()

plt.tight_layout()
plt.subplots_adjust(top=0.92) # Make room for suptitle
plt.savefig(folder_path+f"graph.png", dpi=300, bbox_inches='tight')
plt.show()


##### Best Candidate #####

print("=== Synthesis of Best Candidate After Optimization ===")

# 1) Get best candidate(s) from optimizer
best_candidates = optimizer.best_candidates
best = best_candidates[0]   # first Pareto-optimal candidate
best_vector = torch.from_numpy(best.solution).to(device).float()
best_vector = best_vector.view(phoneme_count, size_per_phoneme)

# 2) Print FITNESS values dynamically
print("Best candidate fitness values:")
for obj_name, score in zip(ACTIVE_OBJECTIVES, best.fitness):
    print(f"  {obj_name}: {score:.6f}")
print()



# 4) Create mixed embeddings using the best vector
# Note: Ensure h_bert_mixed logic matches your loop (sometimes it's fixed to GT, sometimes interpolated)

best_vector = adjustInterpolationVector(best_vector, random_matrix, size_per_phoneme)

# Interpolate Values
h_text_mixed_best = (1.0 - best_vector) * h_text_gt + best_vector * h_text_target
h_bert_mixed_best = h_bert_gt

# 5) Run TTS inference for:
#    - ground-truth
#    - target
#    - best mixed candidate
with torch.no_grad():
    # Ground-truth audio
    audio_gt = tts_model.inference_after_interpolation(
        input_lengths,
        text_mask,
        h_bert_gt,
        h_text_gt,
        style_vector_acoustic,
        style_vector_prosodic,
    )

    # Target audio
    audio_target = tts_model.inference_after_interpolation(
        input_lengths,
        text_mask,
        h_bert_target,
        h_text_target,
        style_vector_acoustic,
        style_vector_prosodic,
    )

    # Best mixed candidate audio
    audio_best = tts_model.inference_after_interpolation(
        input_lengths,
        text_mask,
        h_bert_mixed_best,
        h_text_mixed_best,
        style_vector_acoustic,
        style_vector_prosodic,
    )

# 6) Run ASR on final audio and print predicted text
# Only run if you have the resources/need, but good for validation
asr_final, conf_final = asr_model.analyzeAudio(audio_best)
predicted_text_final = asr_final["text"].strip()

print("ASR Transcription of Best Candidate:")
print(f'  "{predicted_text_final}"')
print(f"  ASR confidence (avg_logprob) = {conf_final}")
print()

# 7) Play all three audios: GT, target, and mixed best
print("=== Ground-truth Audio ===")
display(ipd.Audio(audio_gt, rate=24000))

print("=== Target Audio ===")
display(ipd.Audio(audio_target, rate=24000))

print("=== Best Mixed Candidate Audio ===")

display(ipd.Audio(audio_best, rate=24000))


#### Save Data ####

state_dict = {
    # The Core Result (converted to Tensor for immediate use next time)
    "interpolation_vector": torch.tensor(best.solution).float().cpu(),

    # Metadata (So you know how good it was)
    "fitness_values": best.fitness,
    "active_objectives": ACTIVE_OBJECTIVES,

    # Configuration (Optional: Save text inputs so you can reproduce audio)
    "text_gt": text_gt, # or text_gt depending on your variable name
    "text_target": text_target,
    "asr_text": asr_text,
    "num_generations": num_generations,
    "population_size": pop_size,
    "noise": noise,
    "input_lengths": input_lengths,
    "text_mask": text_mask,
    "h_bert": h_bert_mixed_best,
    "h_text": h_text_mixed_best,


    # Generation info
    "generation_found": getattr(best, "generation", "Unknown") # if your optimizer tracks this
}

torch.save(state_dict, folder_path + f"best_vector.pt")


#### Write Text Summary ####

summary_path = os.path.join(folder_path, "run_summary.txt")

with open(summary_path, "w", encoding="utf-8") as f:
    f.write("=== Adversarial TTS Optimization Summary ===\n\n")

    # ASR text (whatever you stored in asr_text)
    f.write(f"ASR_text: {asr_text}\n\n")

    # Basic config
    f.write(f"Population size (pop_size): {pop_size}\n")
    f.write(f"Number of generations (generation_count): {num_generations}\n")

    # If you want to also log the generation where the best was found (optional)
    f.write(f"Generation best candidate found: {getattr(best, 'generation', 'Unknown')}\n\n")

    # Final fitness values of best candidate
    f.write("Final fitness values (best candidate):\n")
    if len(best.fitness) != len(ACTIVE_OBJECTIVES):
        f.write("  Warning: length mismatch between ACTIVE_OBJECTIVES and best.fitness.\n")
        f.write(f"  Raw fitness values: {best.fitness}\n")
    else:
        for obj_name, score in zip(ACTIVE_OBJECTIVES, best.fitness):
            # ensure it's a plain float for nice formatting
            score_float = float(score)
            f.write(f"  {obj_name}: {score_float:.6f}\n")

    # (Optional) ASR result on final audio
    f.write("\nASR transcription of best candidate:\n")
    f.write(f'  "{predicted_text_final}"\n')
    f.write(f"  ASR confidence (avg_logprob): {float(conf_final):.6f}\n")


#### Write Audio Files ####

sf.write(folder_path + "ground_truth.wav", audio_gt, samplerate=24000)
sf.write(folder_path + "target.wav", audio_target, samplerate=24000)
sf.write(folder_path + "interpolated.wav", audio_best, samplerate=24000)

## Load from state_dict

In [ ]:
loading_folder_path = r"outputs\h_text\utmos_wer_wav2vec-gt\20251202_1010"

In [ ]:
# 1. Load the dictionary from the file
loaded_state_dict = torch.load(os.path.join(loading_folder_path, "best_vector.pt"), weights_only=False)

# 2. Extract the variables back out
interpolation_vector = loaded_state_dict["interpolation_vector"]
h_text_mixed_best = adjustInterpolationVector(h_text_gt, h_text_target, best_vector)
fitness_values = loaded_state_dict["fitness_values"]
active_objectives = loaded_state_dict["active_objectives"]

# Extracting text data
text_gt = loaded_state_dict["text_gt"]
text_target = loaded_state_dict["text_target"]
# asr_text = loaded_state_dict["asr_text"]

# Extracting metadata
generation_found = loaded_state_dict["generation_found"]

# Verification print
print(f"Loaded vector shape: {interpolation_vector.shape}")
# print(f"ASR Text: {asr_text}")
print(f"Fitness: {fitness_values}")

### Load Audio

In [ ]:

y, sr = librosa.load(os.path.join(loading_folder_path, "interpolated.wav"), sr=None) # sr=None keeps the original sample rate

asr_final, conf_final = asr_model.analyzeAudio(y)
predicted_text_final = asr_final["text"].strip()

print(f"ASR Transcription of Best Candidate: {predicted_text_final}")